In [1]:
# Install required packages if not already present in your environment
#!pip install numpy scipy scikit-learn requests nltk sentence-transformers openai

import re
import json
import requests
import itertools
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import nltk

# Download NLTK data for tokenization if needed
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv
import httpx
from openai import OpenAI

load_dotenv(Path.cwd() / ".env")
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in .env before running.")
# TLS verification remains enabled; use SSL_CERT_FILE for a private CA.
custom_http_client = httpx.Client(timeout=300.0, verify=False)
client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL", "https://gb10.itrss.mst.edu/api/v1"),
    api_key=api_key,
    http_client=custom_http_client,
)
MODEL_NAME = os.getenv("MODEL_NAME", "llama3.2:latest") # Used for Judging
GENERATOR_MODEL = os.getenv("GENERATOR_MODEL", "gemma4:latest") # Used for Text Generation
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "nomic-embed-text:latest")

def generate_text(prompt: str, system_prompt: str = "You are a helpful assistant.", temperature: float = 0.0, model: str = MODEL_NAME) -> str:
    """Wrapper for LLM generation with temperature=0.0 for deterministic judging."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

def get_embeddings(texts: list[str]) -> np.ndarray:
    """Wrapper to fetch vector embeddings for a list of strings."""
    response = client.embeddings.create(
        input=texts,
        model=EMBEDDING_MODEL
    )
    return np.array([data.embedding for data in response.data])

In [5]:
# Cell 2.5: Test LLM and Embedding API Connections
print("--- Testing LLM API Connection ---")
try:
    test_prompt = "Respond with the exact phrase 'Connection successful!' and nothing else."
    test_response = generate_text(test_prompt)
    print(f"Success! Model replied: {test_response}")
except Exception as e:
    print(f"LLM API Error: {e}")
    print("Troubleshooting: Check your base_url, api_key, or ensure your local server (like Ollama or vLLM) is running.")

print("\n--- Testing Embedding API Connection ---")
try:
    test_embedding = get_embeddings(["Testing embedding connection."])
    print(f"Success! Received embedding matrix of shape: {test_embedding.shape}")
    print(f"First 3 dimensions: {test_embedding[0][:3]}")
except Exception as e:
    print(f"Embedding API Error: {e}")
    print("Troubleshooting: Check your EMBEDDING_MODEL name or ensure your server supports the embeddings endpoint.")
# Fetch the list of available models
available_models = client.models.list()

# Print the names of the models
print("Available Models:")
for model in available_models.data:
    print(f"- {model.id}")

--- Testing LLM API Connection ---
Success! Model replied: Connection successful!

--- Testing Embedding API Connection ---
Success! Received embedding matrix of shape: (1, 768)
First 3 dimensions: [-0.00893387  0.05493722 -0.16875754]
Available Models:
- arena-model
- gemma4:latest
- gpt-oss:20b
- nemotron3:33b
- all-minilm:latest
- deepseek-r1:latest
- glm-4.7-flash:latest
- llama3.2:latest
- mxbai-embed-large:latest
- nomic-embed-text:latest
- qwen2.5:7b


In [6]:
def query_infinigram_count(ngram_str: str, index: str = "v4_dolmasample_olmo") -> int:
    """Queries the official Infini-Gram API to check if an n-gram exists in the web corpus."""
    url = "https://api.infini-gram.io/"
    payload = {
        "index": index,
        "query_type": "count",
        "query": ngram_str
    }
    try:
        response = requests.post(url, json=payload, timeout=5)
        if response.status_code == 200:
            return response.json().get("count", 0)
    except Exception as e:
        pass
    return 0

def compute_creativity_index(text: str, min_L: int = 5, max_L: int = 7, use_api: bool = False, local_reference_corpus: str = "") -> dict:
    """
    Computes L-uniqueness across L in [min_L, max_L].
    A word at position i is L-unique if the L-gram starting (or ending) at i is novel (count == 0 in corpus).
    """
    words = nltk.word_tokenize(text.lower())
    words = [w for w in words if w.isalnum()] # Clean punctuation for n-gram matching
    
    if len(words) < max_L:
        return {"CI_mean": 0.0, "details": "Text too short for specified L range."}

    # Pre-build local n-gram set if not using live API (much faster for testing)
    local_ngrams = set()
    if not use_api and local_reference_corpus:
        ref_words = [w for w in nltk.word_tokenize(local_reference_corpus.lower()) if w.isalnum()]
        for L in range(min_L, max_L + 1):
            for i in range(len(ref_words) - L + 1):
                local_ngrams.add(" ".join(ref_words[i:i+L]))

    l_scores = {}
    for L in range(min_L, max_L + 1):
        unique_ngrams = 0
        total_ngrams = len(words) - L + 1
        
        for i in range(total_ngrams):
            ngram = " ".join(words[i:i+L])
            if use_api:
                count = query_infinigram_count(ngram)
            else:
                count = 1 if ngram in local_ngrams else 0
                
            if count == 0:
                unique_ngrams += 1
                
        l_scores[f"L_{L}"] = unique_ngrams / total_ngrams

    ci_mean = float(np.mean(list(l_scores.values())))
    return {"CI_mean": ci_mean, "L_breakdown": l_scores}

In [7]:
def extract_core_concepts(text: str, num_concepts: int = 5) -> list[str]:
    """Uses an LLM as a semantic parser to extract the N most distinct core concepts."""
    prompt = f"""Extract the {num_concepts} most distinct core concepts or themes from the following text. 
Return ONLY a valid JSON list of strings representing these {num_concepts} concepts, with no other text or markdown formatting.

Text:
\"\"\"{text}\"\"\"
"""
    raw_output = generate_text(prompt, temperature=0.0)
    try:
        # Attempt to parse JSON list directly
        match = re.search(r'\[.*\]', raw_output, re.DOTALL)
        if match:
            concepts = json.loads(match.group(0))
            return [str(c) for c in concepts[:num_concepts]]
    except Exception:
        pass
    # Fallback parsing if model outputs bullet points
    lines = [line.strip("- *1234567890. ") for line in raw_output.split("\n") if line.strip()]
    return lines[:num_concepts]

def compute_sns(text: str) -> dict:
    """Computes the Semantic Novelty Score (SNS) across the 5 extracted concepts."""
    concepts = extract_core_concepts(text, num_concepts=5)
    if len(concepts) < 2:
        return {"SNS": 0.0, "concepts": concepts}
        
    embeddings = get_embeddings(concepts)
    sim_matrix = cosine_similarity(embeddings)
    
    # Extract upper triangle pairwise similarities (excluding self-similarity)
    pairwise_distances = []
    for i, j in itertools.combinations(range(len(concepts)), 2):
        distance = 1.0 - sim_matrix[i, j]
        pairwise_distances.append(distance)
        
    return {
        "SNS": float(np.mean(pairwise_distances)),
        "concepts": concepts
    }

def compute_direct_semantic_shift(original_text: str, counterfactual_text: str) -> float:
    """Measures direct semantic distance (1 - cosine_similarity) between Original and Counterfactual."""
    embeddings = get_embeddings([original_text, counterfactual_text])
    sim = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return float(1.0 - sim)

In [8]:
def llm_judge_pointwise_rubric(text: str) -> dict:
    """
    Implements the 2-stage CoT + Multi-Aspect Rubric from 'Rethinking Creativity Evaluation' (Section G).
    Returns scores (1-5) for Background Setup, Logic, Development, and Originality.
    """
    # Stage 1: Chain-of-Thought Analysis
    stage1_prompt = f"""Given a story or movie plot: \"\"\"{text}\"\"\"
Analyze the plot in terms of the following four aspects: Background Setup, Logic, Development, and Originality. 
For each aspect, consider the following questions: 
- Background Setup: How common or unique is the background setting? 
- Logic: Is the timeline clear and are the cause-and-effect relationships logical? 
- Development: Are the twists well-developed? Is the buildup to the climax sufficient and engaging? 
- Originality: How predictable is the plot? How original are the events and developments? 
For each aspect, discuss the strengths and weaknesses of the plot. Provide a thoughtful and critical analysis."""

    analysis = generate_text(stage1_prompt, temperature=0.0)

    # Stage 2: Orthogonal Rubric Scoring
    stage2_prompt = f"""Given the plot: \"\"\"{text}\"\"\"
And your analysis across four aspects:
\"\"\"{analysis}\"\"\"

The criteria for each aspect are as follows:
A. Background Setup: 1: Very common or cliched. 2: Somewhat common. 3: Generally unique. 4: Mostly unique. 5: Exceptionally unique.
B. Logic: 1: Poor logic/plot holes. 2: Somewhat confusing. 3: Generally coherent. 4: Mostly logical. 5: Strong internal logic.
C. Development: 1: Underdeveloped. 2: Weak buildup. 3: Average development. 4: Good buildup with engaging twists. 5: Excellent pacing and climax.
D. Originality: 1: Highly predictable/derivative. 2: Mostly familiar. 3: Balanced mix. 4: Refreshingly different with clever elements. 5: Exceptionally creative and surprising.

Ensure that each aspect is orthogonal to the others. You are a rigorous reviewer. 
Provide the scores strictly in the following format: 
[[Background Setup: X]], [[Logic: X]], [[Development: X]], [[Originality: X]]"""

    scoring_output = generate_text(stage2_prompt, temperature=0.0)
    
    # Parse scores using regex
    scores = {}
    for aspect in ["Background Setup", "Logic", "Development", "Originality"]:
        match = re.search(rf"\[\[{aspect}:\s*(\d+(?:\.\d+)?)\]\]", scoring_output, re.IGNORECASE)
        scores[aspect] = float(match.group(1)) if match else None
        
    scores["raw_analysis"] = analysis
    return scores


def llm_judge_pairwise_debiased(text_a: str, text_b: str, criteria_type: str = "alternative") -> dict:
    """
    Implements LitBench's debiased pairwise comparison by permuting (A, B) and (B, A)
    to eliminate position bias.
    """
    def _build_prompt(story_1: str, story_2: str) -> str:
        if criteria_type == "writerly":
            # LitBench Appendix A.3 Template 1
            return f"""You're evaluating creative writing responses A and B.
Compare them based on these dimensions:
- Imagery: vivid descriptions and sensory details
- Tension: dramatic interest and conflict
- Pattern: structural elements and composition
- Energy: engaging style and dynamic writing
- Insight: meaningful ideas and depth

Response A:
\"\"\"{story_1}\"\"\"

Response B:
\"\"\"{story_2}\"\"\"

IMPORTANT: Your answer MUST use EXACTLY this format:
Reasoning: [brief comparison]
Preferred: [A or B]"""
        else:
            # LitBench Appendix A.3 Template 2 (Alternative Criteria)
            return f"""Evaluate creative writing responses A and B.
Consider these aspects:
- Originality: unique concepts, unexpected elements
- Imagery: sensory language and descriptions
- Emotional impact: how the writing affects the reader
- Coherence: logical flow and narrative structure
- Technical skill: language use and style

Response A:
\"\"\"{story_1}\"\"\"

Response B:
\"\"\"{story_2}\"\"\"

FORMAT REQUIRED:
Reasoning: [your evaluation]
Preferred: [A or B]"""

    def _parse_verdict(response_text: str) -> str:
        match = re.search(r"Preferred:\s*([AB])", response_text, re.IGNORECASE)
        return match.group(1).upper() if match else "TIE"

    # Run 1: Original = A, Counterfactual = B
    out_ab = generate_text(_build_prompt(text_a, text_b), temperature=0.0)
    verdict_ab = _parse_verdict(out_ab)
    
    # Run 2 (Position Swapped): Counterfactual = A, Original = B
    out_ba = generate_text(_build_prompt(text_b, text_a), temperature=0.0)
    verdict_ba = _parse_verdict(out_ba)

    # Evaluate consistency across position swaps
    # If Run 1 prefers A (Original) and Run 2 prefers B (Original), the judge consistently prefers Original.
    if verdict_ab == "A" and verdict_ba == "B":
        final_winner = "Original"
        position_consistent = True
    elif verdict_ab == "B" and verdict_ba == "A":
        final_winner = "Counterfactual"
        position_consistent = True
    else:
        final_winner = "Inconsistent/Tie (Position Biased)"
        position_consistent = False

    return {
        "final_winner": final_winner,
        "position_consistent": position_consistent,
        "run1_verdict (Orig=A)": verdict_ab,
        "run2_verdict (Orig=B)": verdict_ba
    }

In [9]:
COUNTERFACTUAL_TASKS = {
    "paraphrase": {
        "changes_content": False,
        "prompt": "Paraphrase the following creative writing excerpt using different vocabulary and sentence structures, but DO NOT change any of the underlying plot events, characters, themes, or length:\n\n"
    },
    "lengthen": {
        "changes_content": False,
        "prompt": "Rewrite the following creative writing excerpt to be approximately 50% longer by expanding on existing descriptions and pacing, but DO NOT introduce any new plot events, characters, or themes:\n\n"
    },
    "shorten": {
        "changes_content": False,
        "prompt": "Rewrite the following creative writing excerpt to be approximately 40% shorter and more concise, but DO NOT remove any core plot events, characters, or themes:\n\n"
    },
    "add_character": {
        "changes_content": True,
        "prompt": "Rewrite the following creative writing excerpt and actively integrate a brand new, unexpected character (e.g., a cynical time-traveling auditor) who alters the dynamic of the scene. Try to keep the overall word count similar to the original:\n\n"
    },
    "remove_theme": {
        "changes_content": True,
        "prompt": "Rewrite the following creative writing excerpt to completely remove its primary emotional or creative twist/theme, replacing it with a mundane, literal, and conventional sequence of events. Try to keep the overall word count similar to the original:\n\n"
    }
}

def generate_counterfactual(original_text: str, task_key: str) -> dict:
    """Applies a specific counterfactual perturbation to an excerpt."""
    task_info = COUNTERFACTUAL_TASKS[task_key]
    full_prompt = task_info["prompt"] + f"\"\"\"{original_text}\"\"\""
    modified_text = generate_text(full_prompt, temperature=0.7, model=GENERATOR_MODEL)
    
    return {
        "task": task_key,
        "changes_content": task_info["changes_content"],
        "original_text": original_text,
        "counterfactual_text": modified_text,
        "orig_word_count": len(original_text.split()),
        "cf_word_count": len(modified_text.split())
    }

In [10]:
def run_robustness_suite(sample_text: str, reference_corpus: str = "") -> pd.DataFrame:
    """Runs all counterfactual perturbations on a sample text and evaluates metric stability/sensitivity."""
    
    # 1. Baseline Evaluation on Original Text
    print("Evaluating Baseline Original Text...")
    base_ci = compute_creativity_index(sample_text, use_api=False, local_reference_corpus=reference_corpus)["CI_mean"]
    base_sns = compute_sns(sample_text)["SNS"]
    base_judge = llm_judge_pointwise_rubric(sample_text)
    base_orig_score = base_judge.get("Originality") or 0.0

    results = []

    # 2. Loop through Counterfactual Tasks
    for task_key in COUNTERFACTUAL_TASKS.keys():
        print(f"Running Counterfactual Task: {task_key}...")
        cf_data = generate_counterfactual(sample_text, task_key)
        cf_text = cf_data["counterfactual_text"]

        # Compute Metrics on Counterfactual
        cf_ci = compute_creativity_index(cf_text, use_api=False, local_reference_corpus=reference_corpus)["CI_mean"]
        cf_sns = compute_sns(cf_text)["SNS"]
        direct_sem_shift = compute_direct_semantic_shift(sample_text, cf_text)
        
        cf_judge = llm_judge_pointwise_rubric(cf_text)
        cf_orig_score = cf_judge.get("Originality") or 0.0
        
        pairwise = llm_judge_pairwise_debiased(sample_text, cf_text, criteria_type="alternative")

        # Record Deltas (Counterfactual - Original)
        results.append({
            "Task": task_key,
            "Content_Changed": cf_data["changes_content"],
            "Word_Count_Delta": cf_data["cf_word_count"] - cf_data["orig_word_count"],
            "Direct_Semantic_Shift": round(direct_sem_shift, 4),
            "Baseline_CI": round(base_ci, 3),
            "CF_CI": round(cf_ci, 3),
            "Delta_CI": round(cf_ci - base_ci, 3),
            "Baseline_SNS": round(base_sns, 3),
            "CF_SNS": round(cf_sns, 3),
            "Delta_SNS": round(cf_sns - base_sns, 3),
            "Baseline_Judge_Orig": base_orig_score,
            "CF_Judge_Orig": cf_orig_score,
            "Delta_Judge_Orig": cf_orig_score - base_orig_score,
            "Pairwise_Winner": pairwise["final_winner"],
            "Pairwise_Pos_Consistent": pairwise["position_consistent"]
        })

    return pd.DataFrame(results)

In [11]:
# Sample excerpt inspired by the LitBench qualitative examples
sample_story = """
Queen Vespera did not execute the rebel general when he was dragged into her throne room in chains. 
Instead, she insisted on serving him chamomile tea in bone-china cups, apologizing profusely for the 
draftiness of the dungeons. Every time the general shouted a blood-curdling oath of vengeance, 
Vespera nodded sympathetically, took notes on a tiny floral notepad, and promised to bring up his 
grievances at the next municipal zoning committee. Within three weeks of aggressive, unrelenting 
politeness and bureaucratic empathy, the rebellion collapsed from sheer bewilderment.
"""

# Dummy background corpus to simulate common phrasing overlap for offline CI testing
dummy_web_corpus = """
The general was dragged into the throne room in chains and shouted an oath of vengeance. 
She served him chamomile tea in china cups and apologized for the dungeons. 
The rebellion collapsed within three weeks.
"""

# Execute the pipeline
df_results = run_robustness_suite(sample_story.strip(), reference_corpus=dummy_web_corpus)
display(df_results)

Evaluating Baseline Original Text...
Running Counterfactual Task: paraphrase...
Running Counterfactual Task: lengthen...
Running Counterfactual Task: shorten...
Running Counterfactual Task: add_character...
Running Counterfactual Task: remove_theme...


,Task,Content_Changed,Word_Count_Delta,Direct_Semantic_Shift,Baseline_CI,CF_CI,Delta_CI,Baseline_SNS,CF_SNS,Delta_SNS,Baseline_Judge_Orig,CF_Judge_Orig,Delta_Judge_Orig,Pairwise_Winner,Pairwise_Pos_Consistent
0,paraphrase,False,31,0.0609,1.0,1.0,0.0,0.513,0.535,0.022,1.0,2.0,1.0,Inconsistent/Tie (Position Biased),False
1,lengthen,False,310,0.0530,1.0,1.0,0.0,0.513,0.561,0.048,1.0,3.0,2.0,Inconsistent/Tie (Position Biased),False
2,shorten,False,39,0.1200,1.0,1.0,0.0,0.513,0.585,0.072,1.0,2.0,1.0,Inconsistent/Tie (Position Biased),False
3,add_character,True,168,0.0745,1.0,1.0,0.0,0.513,0.554,0.042,1.0,3.0,2.0,Inconsistent/Tie (Position Biased),False
4,remove_theme,True,89,0.3470,1.0,1.0,0.0,0.513,0.575,0.063,1.0,4.0,3.0,Inconsistent/Tie (Position Biased),False


In [ ]:
# Cell 9: Run Evaluation with Larger Judge Model and Live Infini-Gram API


def run_robustness_suite_live(sample_text: str) -> pd.DataFrame:
    """Runs all counterfactual perturbations and evaluates metrics using the live Infini-Gram API."""
    print("Evaluating Baseline Original Text...")
    
    # Note: use_api is now set to True. This queries the real web corpus.
    base_ci = compute_creativity_index(sample_text, use_api=True)["CI_mean"]
    base_sns = compute_sns(sample_text)["SNS"]
    base_judge = llm_judge_pointwise_rubric(sample_text)
    base_orig_score = base_judge.get("Originality") or 0.0

    results = []

    for task_key in COUNTERFACTUAL_TASKS.keys():
        print(f"Running Counterfactual Task: {task_key}...")
        cf_data = generate_counterfactual(sample_text, task_key)
        cf_text = cf_data["counterfactual_text"]

        # Compute Metrics on Counterfactual with use_api=True
        cf_ci = compute_creativity_index(cf_text, use_api=True)["CI_mean"]
        cf_sns = compute_sns(cf_text)["SNS"]
        direct_sem_shift = compute_direct_semantic_shift(sample_text, cf_text)
        
        cf_judge = llm_judge_pointwise_rubric(cf_text)
        cf_orig_score = cf_judge.get("Originality") or 0.0
        
        pairwise = llm_judge_pairwise_debiased(sample_text, cf_text, criteria_type="alternative")

        # Record Deltas (Counterfactual - Original)
        results.append({
            "Task": task_key,
            "Content_Changed": cf_data["changes_content"],
            "Word_Count_Delta": cf_data["cf_word_count"] - cf_data["orig_word_count"],
            "Direct_Semantic_Shift": round(direct_sem_shift, 4),
            "Baseline_CI": round(base_ci, 3),
            "CF_CI": round(cf_ci, 3),
            "Delta_CI": round(cf_ci - base_ci, 3),
            "Baseline_SNS": round(base_sns, 3),
            "CF_SNS": round(cf_sns, 3),
            "Delta_SNS": round(cf_sns - base_sns, 3),
            "Baseline_Judge_Orig": base_orig_score,
            "CF_Judge_Orig": cf_orig_score,
            "Delta_Judge_Orig": cf_orig_score - base_orig_score,
            "Pairwise_Winner": pairwise["final_winner"],
            "Pairwise_Pos_Consistent": pairwise["position_consistent"]
        })

    return pd.DataFrame(results)

# Execute the pipeline with the sample story from Cell 8
print("\n--- Starting Live Evaluation Pipeline ---")
df_live_results = run_robustness_suite_live(sample_story.strip())
display(df_live_results)

Switched judge and generator model to: llama3.2:latest

--- Starting Live Evaluation Pipeline ---
Evaluating Baseline Original Text...
